# 24.7 Ant colony optimization

Ant colony optimization builds solutions probabilistically from pheromone memory and local heuristic desirability.

Ant colony optimization is swarm intelligence for combinatorial construction: paths, schedules, assignments, and tours where a solution is assembled one choice at a time.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 2406
rng = np.random.default_rng(SEED)


def sphere_loss(x):
    x = np.asarray(x, dtype=float)
    return float(np.sum(x * x))


def rastrigin_loss(x):
    x = np.asarray(x, dtype=float)
    n = x.size
    waves = x * x - 10.0 * np.cos(2.0 * np.pi * x)
    return float(10.0 * n + np.sum(waves))


def constrained_loss(x):
    x = np.asarray(x, dtype=float)
    base = sphere_loss(x - np.array([1.0, -1.0]))
    violation = max(0.0, x[0] + x[1] - 1.0)
    lower = np.maximum(0.0, -5.0 - x)
    upper = np.maximum(0.0, x - 5.0)
    bounds_penalty = float(np.sum(lower * lower + upper * upper))
    return float(base + 50.0 * violation * violation + 20.0 * bounds_penalty)


def make_black_box_ladder():
    ladder = []
    ladder.append({
        "name": "D1 1-D quadratic",
        "dim": 1,
        "bounds": np.array([[-5.0, 5.0]]),
        "objective": lambda x: float((np.asarray(x)[0] - 3.0) ** 2),
        "optimum": np.array([3.0]),
    })
    ladder.append({
        "name": "D2 2-D sphere",
        "dim": 2,
        "bounds": np.array([[-5.0, 5.0], [-5.0, 5.0]]),
        "objective": sphere_loss,
        "optimum": np.zeros(2),
    })
    ladder.append({
        "name": "D3 multimodal Rastrigin",
        "dim": 2,
        "bounds": np.array([[-5.12, 5.12], [-5.12, 5.12]]),
        "objective": rastrigin_loss,
        "optimum": np.zeros(2),
    })
    ladder.append({
        "name": "D4 constrained penalty",
        "dim": 2,
        "bounds": np.array([[-5.0, 5.0], [-5.0, 5.0]]),
        "objective": constrained_loss,
        "optimum": np.array([1.0, -1.0]),
    })
    ladder.append({
        "name": "D5 high-dim Rastrigin",
        "dim": 30,
        "bounds": np.tile(np.array([[-5.12, 5.12]]), (30, 1)),
        "objective": rastrigin_loss,
        "optimum": np.zeros(30),
    })
    return ladder


def sample_uniform(bounds, count, generator):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    samples = generator.uniform(lows, highs, size=(count, len(bounds)))
    return samples


def clip_to_bounds(x, bounds):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    return np.minimum(np.maximum(x, lows), highs)


def reflect_to_bounds(x, v, bounds):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    below = x < lows
    above = x > highs
    x = np.minimum(np.maximum(x, lows), highs)
    v = np.where(below | above, -0.5 * v, v)
    return x, v


def plot_landscape(ax, rung):
    dim = rung["dim"]
    bounds = rung["bounds"]
    if dim == 1:
        xs = np.linspace(bounds[0, 0], bounds[0, 1], 200)
        ys = np.array([rung["objective"]([x]) for x in xs])
        ax.plot(xs, ys, color="0.35")
        ax.set_xlabel("x")
        ax.set_ylabel("loss")
        return
    grid_x = np.linspace(bounds[0, 0], bounds[0, 1], 80)
    grid_y = np.linspace(bounds[1, 0], bounds[1, 1], 80)
    xx, yy = np.meshgrid(grid_x, grid_y)
    zz = np.zeros_like(xx)
    for row in range(xx.shape[0]):
        for col in range(xx.shape[1]):
            probe = np.zeros(dim)
            probe[0] = xx[row, col]
            probe[1] = yy[row, col]
            zz[row, col] = rung["objective"](probe)
    ax.contourf(xx, yy, zz, levels=24, cmap="viridis")
    ax.set_xlabel("x0")
    ax.set_ylabel("x1")


def make_route_ladder():
    generator = np.random.default_rng(2470)
    ladder = []
    line = np.column_stack([np.linspace(0.0, 1.0, 8), np.zeros(8)])
    ladder.append({"name": "D1 1-D line route", "coords": line, "forbidden": set(), "penalty": 0.0})
    square = np.array([[0.0, 0.0], [1.0, 0.0], [0.7, 0.7], [0.0, 1.0], [0.5, 0.2], [0.2, 0.6], [0.9, 0.4], [0.4, 0.9]])
    ladder.append({"name": "D2 2-D Euclidean routing", "coords": square, "forbidden": set(), "penalty": 0.0})
    cluster_a = generator.normal(loc=[-1.5, 0.0], scale=0.25, size=(8, 2))
    cluster_b = generator.normal(loc=[1.5, 0.0], scale=0.25, size=(8, 2))
    cluster_c = generator.normal(loc=[0.0, 1.8], scale=0.25, size=(8, 2))
    ladder.append({"name": "D3 multimodal clustered TSP", "coords": np.vstack([cluster_a, cluster_b, cluster_c]), "forbidden": set(), "penalty": 0.0})
    constrained = generator.uniform(-2.0, 2.0, size=(18, 2))
    forbidden = {(0, 7), (7, 0), (3, 12), (12, 3), (5, 14), (14, 5)}
    ladder.append({"name": "D4 constrained forbidden-edge route", "coords": constrained, "forbidden": forbidden, "penalty": 20.0})
    high = generator.uniform(-3.0, 3.0, size=(40, 2))
    ladder.append({"name": "D5 high-dim 40-city TSP", "coords": high, "forbidden": set(), "penalty": 0.0})
    return ladder


def distance_matrix(coords):
    diff = coords[:, None, :] - coords[None, :, :]
    dist = np.sqrt(np.sum(diff * diff, axis=2))
    dist = dist + np.eye(len(coords)) * 1.0e9
    return dist


def route_cost(tour, dist, forbidden=None, penalty=0.0):
    if forbidden is None:
        forbidden = set()
    total = 0.0
    for idx in range(len(tour)):
        a = int(tour[idx])
        b = int(tour[(idx + 1) % len(tour)])
        total += float(dist[a, b])
        if (a, b) in forbidden:
            total += penalty
    return total


def plot_route(ax, coords, tour, pheromone=None):
    ax.scatter(coords[:, 0], coords[:, 1], s=20, color="black")
    if tour is not None:
        closed = np.r_[tour, tour[0]]
        ax.plot(coords[closed, 0], coords[closed, 1], color="tab:orange", linewidth=1.4)
    if pheromone is not None:
        n = len(coords)
        cutoff = np.quantile(pheromone[pheromone < np.max(pheromone)], 0.95)
        for i in range(n):
            for j in range(i + 1, n):
                if pheromone[i, j] >= cutoff:
                    ax.plot([coords[i, 0], coords[j, 0]], [coords[i, 1], coords[j, 1]], color="tab:blue", alpha=0.12)
    ax.set_aspect("equal", adjustable="box")


## The concept, built once: pheromone-weighted construction

ACO uses $$P(i\to j)=\frac{\tau_{ij}^{\alpha}\eta_{ij}^{\beta}}{\sum_{k\in A_i}\tau_{ik}^{\alpha}\eta_{ik}^{\beta}},\qquad \tau_{ij}\leftarrow(1-\rho)\tau_{ij}+\Delta\tau_{ij}.$$ We first reproduce the lesson's transition probabilities and pheromone update.

In [ ]:
tau = np.ones(3)
eta = np.array([1.0, 0.25, 1.0 / 9.0])
scores = tau * eta
probs = scores / np.sum(scores)
rounded_probs = [round(float(value), 3) for value in probs]
assert rounded_probs == [0.735, 0.184, 0.082]
old_tau = np.array([1.0, 1.0, 1.0])
rho = 0.2
deposit = np.array([0.5, 0.0, 1.0])
new_tau = (1.0 - rho) * old_tau + deposit
assert np.allclose(new_tau, [1.3, 0.8, 1.8])
assert round(float(new_tau[2] / new_tau[1]), 2) == 2.25
print("probabilities", rounded_probs)
print("pheromone", new_tau)

Now we implement a real ant-colony tour builder: each ant samples a full permutation, pheromone evaporates, deposits scale inversely with tour cost, and pheromone bounds keep exploration alive.

In [ ]:
def ant_colony_optimize(coords, forbidden=None, penalty=0.0, n_ants=24, generations=45, alpha=1.0, beta=2.0, rho=0.2, q=1.0, tau_min=0.02, tau_max=6.0, seed=0):
    generator = np.random.default_rng(seed)
    n = len(coords)
    dist = distance_matrix(coords)
    heuristic = 1.0 / np.maximum(dist, 1.0e-9)
    pheromone = np.ones((n, n))
    best_tour = None
    best_cost = np.inf
    history = []
    best_edges = []
    for generation in range(generations):
        tours = []
        costs = []
        for ant in range(n_ants):
            start = int(generator.integers(0, n))
            tour = [start]
            unvisited = set(range(n))
            unvisited.remove(start)
            while unvisited:
                current = tour[-1]
                choices = np.array(sorted(unvisited), dtype=int)
                scores = (pheromone[current, choices] ** alpha) * (heuristic[current, choices] ** beta)
                scores = np.maximum(scores, 1.0e-12)
                probs = scores / np.sum(scores)
                next_node = int(generator.choice(choices, p=probs))
                tour.append(next_node)
                unvisited.remove(next_node)
            cost = route_cost(tour, dist, forbidden, penalty)
            tours.append(np.array(tour, dtype=int))
            costs.append(cost)
            if cost < best_cost:
                best_cost = float(cost)
                best_tour = np.array(tour, dtype=int)
        pheromone = (1.0 - rho) * pheromone
        for tour, cost in zip(tours, costs):
            deposit = q / max(cost, 1.0e-9)
            for idx in range(n):
                a = int(tour[idx])
                b = int(tour[(idx + 1) % n])
                pheromone[a, b] += deposit
                pheromone[b, a] += deposit
        pheromone = np.minimum(np.maximum(pheromone, tau_min), tau_max)
        history.append(-best_cost)
        best_edges.append(best_tour.copy())
    return {
        "best_tour": best_tour,
        "best_cost": best_cost,
        "best_fitness": -best_cost,
        "history": np.array(history),
        "pheromone": pheromone,
        "best_edges": best_edges,
        "dist": dist,
    }




## The dataset ladder

The route ladder mirrors the F4 difficulty ladder: a 1-D line, a 2-D Euclidean toy, a clustered multimodal TSP, a constrained route with forbidden edges, and a 40-city high-dimensional routing instance.

In [ ]:
route_ladder = make_route_ladder()
for rung in route_ladder:
    coords = rung["coords"]
    print(rung["name"], "cities=", len(coords), "coord_shape=", coords.shape, "forbidden_edges=", len(rung["forbidden"]))
    print("sample", np.round(coords[:3], 3))

In [ ]:
aco_results = []
for idx, rung in enumerate(route_ladder):
    result = ant_colony_optimize(rung["coords"], rung["forbidden"], rung["penalty"], seed=SEED + idx)
    aco_results.append(result)
    print(rung["name"], "best_cost=", round(result["best_cost"], 3), "best_fitness=", round(result["best_fitness"], 3), "convergence_gen=", int(np.argmax(result["history"])))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for idx, rung in enumerate(route_ladder):
    ax = axes[0, idx]
    plot_route(ax, rung["coords"], aco_results[idx]["best_tour"], aco_results[idx]["pheromone"])
    ax.set_title(rung["name"])
summary_ax = axes[1, 0]
for idx, result in enumerate(aco_results):
    summary_ax.plot(result["history"], label=f"D{idx + 1}")
summary_ax.set_title("best fitness vs generation")
summary_ax.set_xlabel("generation")
summary_ax.set_ylabel("negative tour cost")
summary_ax.legend()
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on D5: no evaporation freezes early pheromone

With `rho=0`, old deposits persist forever. The fix uses evaporation, pheromone bounds, and a nonzero exploration floor.

In [ ]:
d5 = route_ladder[-1]
frozen = ant_colony_optimize(d5["coords"], rho=0.0, tau_min=0.0, tau_max=100.0, seed=77)
fixed = ant_colony_optimize(d5["coords"], rho=0.25, tau_min=0.02, tau_max=4.0, seed=77)
frozen_spread = float(np.max(frozen["pheromone"]) / np.maximum(np.min(frozen["pheromone"]), 1.0e-9))
fixed_spread = float(np.max(fixed["pheromone"]) / np.maximum(np.min(fixed["pheromone"]), 1.0e-9))
print("frozen best fitness", round(frozen["best_fitness"], 3), "pheromone spread", round(frozen_spread, 1))
print("fixed best fitness", round(fixed["best_fitness"], 3), "pheromone spread", round(fixed_spread, 1))

## Evaluate it + Practice

- Metric: best fitness is negative tour cost; convergence generation is the first generation of the best tour.
- Baseline: compare with random permutations using the same number of tours.
- Sanity check: D1 should recover a mostly monotone line tour.
- Ablation: set `rho=0` or set `beta` extremely high and observe lock-in or greediness.
- Failure signals: one edge has huge pheromone while tour quality stops improving.

Practice: change `alpha` and `beta` and plot the transition probabilities from one city.

Practice: add one forbidden edge to D2 and verify the penalty changes the best route.